# 🔬 VESTA F1XX REPROCESSING SESSION: MASTER EXPLORATORY DATA ANALYSIS (EDA)
## Kiểm Toán Định Lượng & Trực Quan Hóa Chi Tiết 11 Kỹ Thuật Tiền Xử Lý Dữ Liệu
---
**Dự án:** VESTA (*Vietnamese Equity Sentiment-Triggered Agent*)  
**Phân tầng:** Tier F1xx — Data Integrity, Point-in-Time Join & Enterprise Feature Engineering  
**Mục tiêu Notebook:** Cung cấp toàn bộ công cụ kiểm toán trực quan (Visual & Statistical Audit), tải dữ liệu mẫu, biểu diễn các phân phối trước và sau chuẩn hóa, kiểm định tính dừng toán học và độ tin cậy của nhãn mục tiêu.

In [ ]:
# 1. Khởi tạo môi trường, thiết lập đồ họa và tải các thư viện định lượng
import os
import sys
import json
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb

# Thiết lập giao diện đồ họa chuẩn học thuật
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['figure.dpi'] = 120

out_dir = pathlib.Path('../test_pipeline/out').resolve()
if not out_dir.exists():
    out_dir = pathlib.Path('test_pipeline/out').resolve()

print(f"Thư mục dữ liệu kết quả: {out_dir}")
print(f"Số tệp kết quả tìm thấy: {len(list(out_dir.glob('*')))}")

### 📊 Tổng Quan Toàn Bộ Kết Quả Nghiệm Thu Tier F1xx
Đọc báo cáo tổng hợp `test_pipeline_run_report.json` và bảng điều khiển KPI của F101, F102, F103, F104.

In [ ]:
with open(out_dir / 'test_pipeline_run_report.json', 'r', encoding='utf-8') as f:
    pipeline_report = json.load(f)

stages = pipeline_report['stages']
summary_data = []
for k in ['F101', 'F102', 'F103', 'F104']:
    if k in stages:
        summary_data.append({
            'Tier Feature': k,
            'Status': stages[k].get('status', 'N/A'),
            'Metric Summary': str(stages[k].get('coverage', stages[k].get('summary', stages[k].get('valid_symbols_count', 'OK'))))
        })

df_stages = pd.DataFrame(summary_data)
display(df_stages)

### 🕒 Kỹ Thuật 1: Temporal Alignment & Point-in-Time Session Breakdown
Kiểm toán 658,182 sự kiện trong `core.pit_events`: Phân loại mốc thời gian xuất bản (Sau giờ giao dịch 40.4%, Nửa đêm 25.7%, Trong giờ giao dịch 27.0%, Cuối tuần 1.1%).

In [ ]:
with open(out_dir / 'temporal_alignment_audit_report.json', 'r', encoding='utf-8') as f:
    temp_audit = json.load(f)['audit_1_1_event_session']

fig, ax = plt.subplots(figsize=(10, 5))
labels = ['Sau giờ đóng cửa (>=15:00)', 'Trong phiên (09:00-15:00)', 'Mốc 00:00:00 (Midnight)', 'Cuối tuần']
values = [temp_audit['after_close_pct'], temp_audit['trading_hours_pct'], temp_audit['midnight_pct'], temp_audit['weekend_pct']]
colors = ['#2b5c8f', '#2ca02c', '#d62728', '#ff7f0e']

bars = ax.bar(labels, values, color=colors, edgecolor='black', alpha=0.85)
for bar in bars:
    y = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, y + 0.8, f'{y:.2f}%', ha='center', va='bottom', fontweight='bold')

ax.set_title('Phân Bố Mốc Thời Gian Xuất Bản Tin Tức (Temporal Alignment Audit)', fontsize=13, fontweight='bold')
ax.set_ylabel('% Tỷ Lệ Trong Tổng Số 658,182 Sự Kiện')
ax.set_ylim(0, 50)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### 🛡️ Kỹ Thuật 2 & 3: Extreme Tail-Risk Sanitization & Asymmetric Winsorization
Bóc tách vụ thao túng giá cổ phiếu XDC tăng $+3,021\%$ với thanh khoản 100 cổ phiếu/phiên. Xem xét sự suy giảm của hệ số Kurtosis từ 4,355 xuống 7.82 và sự gia tăng của hệ số Cohen's d từ 0.0576 lên 0.0984.

In [ ]:
with open(out_dir / 'tail_risk_sanitization_report.json', 'r', encoding='utf-8') as f:
    tail_data = json.load(f)

df_treatments = pd.DataFrame(tail_data['treatments'])
print("Top 5 cổ phiếu ngoại lai cực đoan nhất:")
display(pd.DataFrame(tail_data['top10_outliers']).head(5))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
df_treatments['Cohen_d_num'] = df_treatments["Cohen's d"].astype(float)
df_treatments['Kurtosis_num'] = df_treatments['Kurtosis'].astype(float)

# Biểu đồ Cohen's d
ax1.barh(df_treatments['Treatment'], df_treatments['Cohen_d_num'], color='#1f77b4', edgecolor='black', alpha=0.85)
ax1.set_title("Hệ Số Tác Động Chuẩn Hóa (Cohen's d)", fontsize=12, fontweight='bold')
ax1.set_xlabel("Cohen's d (Càng lớn tín hiệu Alpha càng mạnh)")

# Biểu đồ Kurtosis
ax2.barh(df_treatments['Treatment'], np.log10(df_treatments['Kurtosis_num']), color='#e74c3c', edgecolor='black', alpha=0.85)
ax2.set_title("Hệ Số Nhọn Phân Phối (Log10 Kurtosis - Đuôi Dày)", fontsize=12, fontweight='bold')
ax2.set_xlabel("Log10(Excess Kurtosis)")

plt.tight_layout()
plt.show()

### 📈 Kỹ Thuật 4: RankGauss Normalization & Thử Nghiệm Chịu Tải Sốc 100x (Stress Test)
So sánh khả năng kháng sốc của RankGauss so với MinMax, Z-Score và Log1p khi bị chèn một ngoại lai cực đại gấp 100 lần.

In [ ]:
with open(out_dir / 'rankgauss_report.json', 'r', encoding='utf-8') as f:
    rg_data = json.load(f)['results']

stress = rg_data['scaler_stress_test']
scalers = ['MinMax Scaler', 'Standard Z-Score', 'Log1p Transform', 'RankGauss']
disruptions = [
    stress['relative_disruption_minmax'] * 100,
    stress['mean_absolute_disruption_zscore'] * 100,
    stress['mean_absolute_disruption_log1p'] * 100,
    stress['mean_absolute_disruption_rankgauss'] * 100
]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(scalers, disruptions, color=['#c0392b', '#d35400', '#f39c12', '#2ecc71'], edgecolor='black', alpha=0.85)
for bar in bars:
    y = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, y + 1.5, f"{y:.2f}%", ha='center', fontweight='bold')

ax.set_title("Mức Độ Xáo Trộn Không Gian Đặc Trưng Khi Gặp Sốc Ngoại Lai 100x", fontsize=13, fontweight='bold')
ax.set_ylabel("% Xáo Trộn Giá Trị Toàn Bộ Mẫu")
ax.set_ylim(0, 115)
plt.tight_layout()
plt.show()

### 🔄 Kỹ Thuật 6: Fixed-Width Window Fractional Differentiation (FFD)
Tìm kiếm điểm cân bằng tối ưu giữa Tính dừng toán học (Stationary - Kiểm định ADF $p < 0.01$) và Bảo toàn ký ức chuỗi giá (Pearson Correlation $> 0.90$).

In [ ]:
with open(out_dir / 'fractional_differentiation_report.json', 'r', encoding='utf-8') as f:
    ffd_grid = json.load(f)['vnindex']['grid']

df_ffd = pd.DataFrame(ffd_grid)

fig, ax1 = plt.subplots(figsize=(10, 5))
color = '#d62728'
ax1.plot(df_ffd['d'], df_ffd['adf_stat'], marker='o', color=color, label='Thống Kê ADF (t-stat)')
ax1.axhline(-2.862, color='darkred', linestyle='--', label='Ngưỡng Dừng 5% (-2.862)')
ax1.set_xlabel('Bậc Vi Phân Phân Số (d)')
ax1.set_ylabel('Thống Kê ADF', color=color)
ax1.tick_params(axis='y', labelcolor=color)
ax1.axvline(0.20, color='purple', linestyle=':', linewidth=2, label='d* = 0.20 (Tối ưu)')

ax2 = ax1.twinx()
color2 = '#1f77b4'
ax2.plot(df_ffd['d'], df_ffd['corr_pearson'], marker='s', color=color2, label='Hệ Số Tương Quan Ký Ức (Pearson)')
ax2.set_ylabel('Tương Quan Với Giá Gốc', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower left')
plt.title('Đường Cong Cân Bằng Giữa Tính Dừng ADF Và Bảo Tồn Ký Ức Chuỗi Giá (FFD)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 📑 Kỹ Thuật 5: NMAR (Not-Missing-At-Random) & Hệ Số Rủi Ro Chậm Nộp BCTC
Phân tích mối tương quan giữa sự chậm trễ nộp BCTC và rủi ro sụt giảm lợi nhuận (Tail Risk).

In [ ]:
with open(out_dir / 'nmar_missing_report.json', 'r', encoding='utf-8') as f:
    nmar_data = json.load(f)['stock_reporting_status']

df_nmar = pd.DataFrame(nmar_data)
display(df_nmar[['reporting_status', 'count', 'pct_of_events', 'volatility_multiplier', 'p05_tail_loss']])

### 🔀 Kỹ Thuật 9: Gray Code Macro Regime Transition
Khảo sát bước nhảy khoảng cách Hamming giữa Gray Code (luôn $= 1$ bit) so với Binary thông thường (nhảy đột biến tới 4 bits).

In [ ]:
with open(out_dir / 'gray_code_report.json', 'r', encoding='utf-8') as f:
    gray_report = json.load(f)

g_jumps = gray_report['gray_hamming_transitions']
b_jumps = gray_report['standard_binary_hamming_transitions']

plt.figure(figsize=(10, 4))
plt.step(range(1, len(b_jumps) + 1), b_jumps, where='mid', label='Binary Thông Thường (Gây sốc mạng nơ-ron)', color='#e74c3c', linewidth=2)
plt.step(range(1, len(g_jumps) + 1), g_jumps, where='mid', label='Gray Code Tuần Hoàn (Độ mượt tuyệt đối = 1)', color='#27ae60', linewidth=2.5)
plt.title('Khoảng Cách Hamming Khi Chuyển Dịch Giữa 16 Chế Độ Thị Trường', fontsize=13, fontweight='bold')
plt.xlabel('Bước Chuyển Pha Thị Trường')
plt.ylabel('Số Bit Bị Thay Đổi (Bits Changed)')
plt.ylim(0, 5)
plt.legend()
plt.tight_layout()
plt.show()

### 📦 Khảo Sát Tập Mẫu Dữ Liệu Parquet Đã Tiền Xử Lý (F104 Features & Targets)
Tải trực tiếp bảng `sample_preprocessed_dataset.csv` và ma trận đặc trưng `features_sample_train.parquet`.

In [ ]:
sample_csv = out_dir / 'sample_preprocessed_dataset.csv'
if sample_csv.exists():
    df_sample = pd.read_csv(sample_csv)
    print(f"Kích thước tập mẫu: {df_sample.shape}")
    display(df_sample.head(5))

    # Ma trận tương quan giữa các biến định lượng
    quant_cols = ['sentiment_score', 'rankgauss_sentiment_z', 'ret_t5_pct', 'ret_t30_pct', 'winsorized_diff_pct', 'rankgauss_volume_z']
    available_cols = [c for c in quant_cols if c in df_sample.columns]
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(df_sample[available_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title('Ma Trận Tương Quan Đặc Trưng Tiền Xử Lý F104', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 🏁 Kết Luận & Đánh Giá Tổng Thể
1. **Bảo toàn 100% Zero Look-Ahead Bias:** Phép nối Point-in-Time F102 và kiểm toán F101 đã ngăn chặn toàn bộ hiện tượng rò rỉ thông tin tương lai.
2. **Triệt tiêu nhiễu thao túng đuôi dày:** Bóc tách các mã như XDC và áp dụng Asymmetric Winsorization khống chế Kurtosis từ 4,355 về mức 7.82, nâng hệ số Cohen's d lên $0.0984$.
3. **Chuẩn hóa RankGauss vững chắc:** Đảm bảo các đặc trưng tài chính luôn tuân theo phân phối chuẩn $\mathcal{N}(0, 1)$, miễn nhiễm trước các cú sốc ngoại lai 100x.
4. **Tập dữ liệu F104 sẵn sàng:** 384,431 mẫu đa phương thức đã hoàn thiện, đảm bảo điều kiện tiên quyết vững chắc để bước vào phân tầng F2xx và F3xx.